In [10]:
import pandas as pd
from pathlib import Path
from scipy.stats import mannwhitneyu

path_students = Path("../../data/processed/student_housing_nrw_clean.csv")
path_groups = Path("../../data/processed/city_price_groups.csv")

df_students = pd.read_csv(path_students)
df_groups = pd.read_csv(path_groups)

df_students.shape, df_groups.shape, df_students.columns, df_groups.columns


((17, 6),
 (54, 3),
 Index(['hochschulort', 'wohnheime_2025_anzahl', 'wohnheimplaetze_2025',
        'studierende_ws_2024_2025', 'studierende_je_wohnheimplatz_2025',
        'wohnheimplatzrelation'],
       dtype='object'),
 Index(['city', 'median_price_per_sqm', 'price_group'], dtype='object'))

In [11]:
def clean_city(s: pd.Series) -> pd.Series:
    return (
        s.astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

df_students["hochschulort_clean"] = clean_city(df_students["hochschulort"])
df_groups["city_clean"] = clean_city(df_groups["city"])

df_students[["hochschulort", "hochschulort_clean"]].head(), df_groups[["city", "city_clean", "price_group"]].head()


(  hochschulort hochschulort_clean
 0       Aachen             Aachen
 1    Bielefeld          Bielefeld
 2       Bochum             Bochum
 3         Bonn               Bonn
 4     Dortmund           Dortmund,
            city    city_clean price_group
 0        Aachen        Aachen       teuer
 1  Aachen_Kreis  Aachen_Kreis       teuer
 2     Bielefeld     Bielefeld       teuer
 3        Bochum        Bochum       teuer
 4          Bonn          Bonn       teuer)

In [12]:
df_join = df_students.merge(
    df_groups[["city_clean", "price_group"]],
    left_on="hochschulort_clean",
    right_on="city_clean",
    how="left"
)

# Wie viele NRW-Hochschulorte konnten gematcht werden?
match_rate = df_join["price_group"].notna().mean()
df_join[["hochschulort", "price_group"]].sort_values("hochschulort").head(20), match_rate


(       hochschulort price_group
 0            Aachen       teuer
 1         Bielefeld       teuer
 2            Bochum       teuer
 3              Bonn       teuer
 4          Dortmund       teuer
 5          Duisburg    guenstig
 6        Düsseldorf       teuer
 7             Essen       teuer
 8     Gelsenkirchen    guenstig
 9             Hagen    guenstig
 11          Krefeld       teuer
 10             Köln       teuer
 12  Mönchengladbach       teuer
 13          Münster       teuer
 14        Paderborn         NaN
 15           Siegen         NaN
 16        Wuppertal    guenstig,
 0.8823529411764706)

In [13]:
# Wir testen die Versorgung: kleinere Werte bei "studierende_je_wohnheimplatz_2025" bedeuten bessere Versorgung.
# Alternativ könntest du "wohnheimplatzrelation" testen, falls du die definiert hast.

col_metric = "studierende_je_wohnheimplatz_2025"

df_test = df_join.dropna(subset=["price_group", col_metric]).copy()

df_test["price_group"].value_counts(), df_test[[col_metric]].describe()


(price_group
 teuer       11
 guenstig     4
 Name: count, dtype: int64,
        studierende_je_wohnheimplatz_2025
 count                          15.000000
 mean                           19.066667
 std                            15.736521
 min                             9.000000
 25%                            10.000000
 50%                            14.000000
 75%                            21.000000
 max                            70.000000)

In [14]:
expensive = df_test[df_test["price_group"] == "teuer"][col_metric]
cheap = df_test[df_test["price_group"] == "guenstig"][col_metric]

len(expensive), len(cheap), expensive.median(), cheap.median()


(11, 4, 11.0, 21.5)

In [15]:
u_stat, p_value = mannwhitneyu(expensive, cheap, alternative="two-sided")
u_stat, p_value


(7.5, 0.06586409657914348)

In [16]:
alpha = 0.05
if p_value < alpha:
    print("H₀ wird verworfen: Die Gruppen unterscheiden sich signifikant.")
else:
    print("H₀ kann nicht verworfen werden: Kein signifikanter Unterschied.")


H₀ kann nicht verworfen werden: Kein signifikanter Unterschied.


In [17]:
import numpy as np
from scipy.stats import norm

n1, n2 = len(expensive), len(cheap)
mean_u = n1*n2/2
std_u = np.sqrt(n1*n2*(n1+n2+1)/12)

z = (u_stat - mean_u) / std_u
r = abs(z) / np.sqrt(n1 + n2)

z, r


(-1.893094508518214, 0.4887949002858505)

In [18]:
expensive = df_test[df_test["price_group"] == "teuer"]["studierende_je_wohnheimplatz_2025"]
cheap = df_test[df_test["price_group"] == "guenstig"]["studierende_je_wohnheimplatz_2025"]

expensive.median(), cheap.median()



(11.0, 21.5)

### Wichtig

In [19]:
from scipy.stats import mannwhitneyu

u_stat, p_value = mannwhitneyu(
    expensive,
    cheap,
    alternative="two-sided"
)

u_stat, p_value


(7.5, 0.06586409657914348)

In [20]:
alpha = 0.05

if p_value < alpha:
    print("H₀ wird verworfen: Die Wohnheimversorgung unterscheidet sich signifikant zwischen teuren und günstigen Städten.")
else:
    print("H₀ kann nicht verworfen werden: Kein signifikanter Unterschied zwischen den Gruppen.")


H₀ kann nicht verworfen werden: Kein signifikanter Unterschied zwischen den Gruppen.


In [21]:
import numpy as np

n1, n2 = len(expensive), len(cheap)
mean_u = n1 * n2 / 2
std_u = np.sqrt(n1 * n2 * (n1 + n2 + 1) / 12)

z = (u_stat - mean_u) / std_u
r = abs(z) / np.sqrt(n1 + n2)

z, r


(-1.893094508518214, 0.4887949002858505)

Teure Städte haben bessere Wohnheimversorgung (weniger Studierende pro Platz).

Mann-Whitney-U 

U = 7.5

p = 0.0659

α = 0.05

H₀ kann nicht verworfen werden (knapp oberhalb der Signifikanzgrenze).

Effektgröße

z = −1.89

r = 0.49 → großer Effekt

Obwohl der Test formal nicht signifikant ist, zeigt die Effektgröße einen starken praktischen Unterschied.